# Laya medical mid — Laya recipe, LR×5, 5 epochs (~50k)

`lr_enc=1.25e-4`, `lr_head=5e-4`, CE×1, soft teacher. Attach `wandb_api_key` if available.

**Settings:** GPU T4 x2, Internet on.

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
print('python', sys.version)
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'n_gpu', torch.cuda.device_count())
if torch.cuda.device_count() < 1:
    raise SystemExit('No GPU. Set Accelerator to GPU T4 x2.')
os.environ['USE_TF'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
try:
    from kaggle_secrets import UserSecretsClient
    c = UserSecretsClient()
    for name in ('wandb_api_key', 'WANDB_API_KEY', 'wandb'):
        try:
            v = c.get_secret(name)
        except Exception:
            continue
        if v:
            os.environ['WANDB_API_KEY'] = v
            print('wandb secret loaded_from:', name)
            break
except Exception as e:
    print('kaggle_secrets:', type(e).__name__, e)
print('WANDB_API_KEY present:', bool(os.environ.get('WANDB_API_KEY')))

In [ ]:
%pip -q install -U 'laya>=0.3.7' 'transformers>=4.48.0' 'datasets>=3.0.0' safetensors huggingface_hub accelerate scipy pyarrow pandas tabulate PyYAML wandb

In [ ]:
REPO = 'https://github.com/Priyanshu-5257/laya-medical-finetune.git'
REPO_DIR = Path('/kaggle/working/repo')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.check_call(['git', 'clone', '--branch', 'main', '--depth', '1', REPO, str(REPO_DIR)])
subprocess.check_call(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'])

In [ ]:
def run(cmd, cwd=None, env=None):
    print('+', *cmd, flush=True)
    e = os.environ.copy()
    if env: e.update(env)
    p = subprocess.Popen(cmd, cwd=cwd, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    if rc != 0:
        raise SystemExit(rc)

run(['bash', 'scripts/run_mid_laya_recipe_lr5x.sh'], cwd=str(REPO_DIR), env={
    'WORK': '/kaggle/working',
    'CONFIG': 'configs/mid_laya_recipe_lr5x.yaml',
})

In [ ]:
p = Path('/kaggle/working/summary_mid_laya_lr5x.json')
print(p.read_text() if p.exists() else 'MISSING summary_mid_laya_lr5x.json')